In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]  # importing functions from other folders

sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
from _data.data_utils import read_in
from _fitting.fitting_utils import abbrev_stat
# from _fitting.fitting_utils import hist_plot, CI_plot, CI_plot_alt, CI_plot_both, plot_posteriors_side_by_side, plot_spline_Bknots
import pymc as pm
import pymc.math as pmm
import arviz as az
from patsy import dmatrix
import nutpie
import time
from IPython.display import display
from pymc.variational.callbacks import CheckParametersConvergence
import io
import base64
import re
import pytensor.tensor as pt
from pytensor.gradient import disconnected_grad
from scipy.sparse.linalg import eigsh

az.style.use("arviz-darkgrid")

if '___laptop' in os.listdir('../../'):

    # laptop folder
    folder = "../../../_data/p-dengue/"
elif '___server' in os.listdir('../../'):

    # server folder
    folder = "../../../../../../data/lucaratzinger_data/p_dengue/"

%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
model_fits_folder = os.path.join(folder, "model_fits/")
run_folders = ["a2_201601_201912[smooth_s1|6_selection]"]

---

## Pareto k plots for all variables

In [3]:
def pareto_k_scatter(model_fits_dir, run_folders, p=None, num_knots=None, stat_name=None):

    data_settings = {'admin':2, 'max_lag':6, 'start_year':2016, 'start_month':1, 'end_year':2019, 'end_month':12}
    data = read_in(folder, **data_settings, standardise=True, dropna=True, celsius=True, tp_log=True)
    temp = data['t2m_mean_pop_weighted(0)'].values

    def parse_name(name):
        p_match = re.search(r'p=([\d.]+)', name)
        k_match = re.search(r'__knots\((\d+)', name)
        # s_match = re.search(r'std=(True|False)', name)
        stat_match = re.search(r'\[([a-zA-Z0-9_()]+)\]', name)
        #print(stat_match.group(1) if stat_match else None)
        return {
            'p': float(p_match.group(1)) if p_match else None,
            'num_knots': int(k_match.group(1)) if k_match else None,
            # 'std': s_match.group(1) == 'True' if s_match else None,
            'stat_name': stat_match.group(1) if stat_match else None,
        }

    def to_set(val):
        if val is None: return None
        return set(val) if hasattr(val, '__iter__') else {val}

    p_set, k_set, stat_set = to_set(p), to_set(num_knots), to_set(stat_name)

    matches = []
    for run in run_folders:
        metrics_dir = os.path.join(model_fits_dir, run, 'metrics')
        if not os.path.isdir(metrics_dir):
            continue
        for fname in sorted(os.listdir(metrics_dir)):
            if not fname.endswith('.npz'):
                continue
            meta = parse_name(fname)
            if p_set is not None and meta['p'] not in p_set:
                # print('p')
                continue
            if k_set is not None and meta['num_knots'] not in k_set:
                # print('k')
                continue
            # if std_set is not None and meta['std'] not in std_set:
                # continue
            if stat_set is not None and meta['stat_name'] not in stat_set:
                 #print('stat')
                continue
            pareto_k = np.load(os.path.join(metrics_dir, fname))['pareto_k']
            matches.append((meta, pareto_k))
    
    if not matches:
        raise ValueError("No models matched the given filters.")

    colors = cm.tab20(np.linspace(0, 1, len(matches)))
    fig, ax = plt.subplots(figsize=(10, 4))

    matches.sort(key=lambda x: x[0]['num_knots'])
    for (meta, pareto_k), color in zip(matches, colors):
        label = f"p={meta['p']}, k={meta['num_knots']}, stat={meta['stat_name']}"
        ax.scatter(temp, pareto_k, s=0.5, color=color, label=label, alpha=0.6)

    ax.axhline(0.7, color='red', linestyle='--', linewidth=1, label='Pareto k=0.7 threshold (bad)')
    ax.axhline(1.0, color='red', linestyle='-', linewidth=1, label='Pareto k=1.0 threshold (very bad)')
    ax.set_xlabel(meta['stat_name'] if meta['stat_name'] else 'All')
    ax.set_ylabel('pareto_k')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', markerscale=5)

    return fig

In [4]:
data_settings = {'admin':2, 'max_lag':6, 'start_year':2016, 'start_month':1, 'end_year':2019, 'end_month':12}
_data = read_in(folder, **data_settings, standardise=True, dropna=True, celsius=True, tp_log=True)
statistics = _data.columns.tolist()
# that start with t2m, rh, tp
statistics = [name for name in statistics if name.startswith(('t2m','rh','tp'))]
# that contains 'pop_weighted'
statistics = [name for name in statistics if 'pop_weighted' in name]
# if it contains 'tp' then it should contain 'log('
statistics = [name for name in statistics if not (name.startswith('tp') and 'log(' not in name)]
lags = [0, 1, 3, 5]
statistics = [stat for stat in statistics if any(f"({lag})" in stat for lag in lags)]
statistics.sort()

In [ ]:
p_vals = [1.0, 2.5, 5.0, 7.5, 10.0, 12.5, 16.0, 20.0, 25.0, 1000.0]
num_knots_list = [5, 10, 15, 20, 25, 30, 35, 40, 50, 60]
print(statistics)
# stat_name_list = ['rh_mean_p(0)']

for s_name in statistics:
    stat_name_list = [abbrev_stat(s_name)]
    print(stat_name_list)
    html_parts = ['<html><body>']
    for p in p_vals:
        fig = pareto_k_scatter(model_fits_folder, run_folders, p=[p], num_knots=num_knots_list, stat_name=stat_name_list)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        b64 = base64.b64encode(buf.read()).decode('utf-8')
        html_parts.append(f'<h3>p={p}</h3><img src="data:image/png;base64,{b64}"><br>')

    html_parts.append('</body></html>')

    with open(f'pareto_k_values/pareto_k[{stat_name_list}].html', 'w') as f:
        f.write('\n'.join(html_parts))

['rh_mean_pop_weighted(0)', 'rh_mean_pop_weighted(1)', 'rh_mean_pop_weighted(3)', 'rh_mean_pop_weighted(5)', 't2m_max_pop_weighted(0)', 't2m_max_pop_weighted(1)', 't2m_max_pop_weighted(3)', 't2m_max_pop_weighted(5)', 't2m_mean_pop_weighted(0)', 't2m_mean_pop_weighted(1)', 't2m_mean_pop_weighted(3)', 't2m_mean_pop_weighted(5)', 't2m_min_pop_weighted(0)', 't2m_min_pop_weighted(1)', 't2m_min_pop_weighted(3)', 't2m_min_pop_weighted(5)', 'tp_24hmax_pop_weighted_log(0)', 'tp_24hmax_pop_weighted_log(1)', 'tp_24hmax_pop_weighted_log(3)', 'tp_24hmax_pop_weighted_log(5)', 'tp_24hmean_pop_weighted_log(0)', 'tp_24hmean_pop_weighted_log(1)', 'tp_24hmean_pop_weighted_log(3)', 'tp_24hmean_pop_weighted_log(5)']
['rh_mean_p(0)']
['rh_mean_p(1)']
['rh_mean_p(3)']
['rh_mean_p(5)']
['t2m_max_p(0)']
['t2m_max_p(1)']
['t2m_max_p(3)']
['t2m_max_p(5)']
['t2m_mean_p(0)']
['t2m_mean_p(1)']
['t2m_mean_p(3)']


---